# 01 — Exploratory Data Analysis (EDA): ASX OHLCV Daily

This notebook is the **EDA** companion for the ASX OHLCV daily dataset. It is deliberately structured to be:
- Methodologically defensible (clear sequencing and decisions)
- Re-runnable (parameterised data access)
- Lightweight (minimal narrative, high signal)

**Scope note:** This notebook focuses on *understanding* the data. Any *transformations that change the dataset* are deferred to `02_preprocessing.ipynb`.


## Purpose
- Understand schema, coverage, and quality of the ASX OHLCV dataset
- Characterise distributions and relationships (univariate → bivariate → multivariate)
- Identify anomalies and potential outliers **without** applying treatment yet
- Produce a concise set of findings and implications for downstream steps


## Dataset Overview and Scope
- Each row represents **one ticker on one trade date** at **daily** granularity
- Dataset contains OHLCV measures and associated metadata (vendor, ingestion context)
- This EDA assumes the dataset is already curated to a consistent schema (e.g., via pipeline)
- Corporate actions (splits/dividends) are not explicitly modelled here unless present as fields


In [ ]:
# Core imports (standard)
import os
import io
from dataclasses import dataclass
from typing import Optional, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


## Data Access Configuration (MinIO)

This notebook supports reading Parquet from MinIO using the `minio` Python client.

If you are running this notebook **inside the same Docker Compose network** as MinIO, the default endpoint
`minio:9000` should work. If you are running locally, you may need `localhost:9000` (or a mapped port).


In [ ]:
# Optional dependency: minio
# If you are running this notebook in a fresh environment, uncomment the install line.
# %pip -q install minio pyarrow

from minio import Minio


In [ ]:
@dataclass(frozen=True)
class MinIOConfig:
    endpoint: str = os.environ.get("MINIO_ENDPOINT", "minio:9000")
    access_key: str = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
    secret_key: str = os.environ.get("MINIO_SECRET_KEY", "minioadmin")
    secure: bool = os.environ.get("MINIO_SECURE", "false").lower() == "true"

CFG = MinIOConfig()

# Dataset location (override via env vars if required)
BUCKET = os.environ.get("ASX_BUCKET", "curated")
PREFIX  = os.environ.get("ASX_PREFIX", "tabular/market_ohlcv_daily/exchange=ASX/")

# Optional safety control for large datasets
MAX_FILES = int(os.environ.get("ASX_MAX_FILES", "0"))  # 0 means load all files found under prefix


In [ ]:
def minio_client(cfg: MinIOConfig) -> Minio:
    return Minio(
        cfg.endpoint,
        access_key=cfg.access_key,
        secret_key=cfg.secret_key,
        secure=cfg.secure,
    )

def list_parquet_objects(client: Minio, bucket: str, prefix: str, max_files: int = 0) -> List[str]:
    objs = []
    for obj in client.list_objects(bucket, prefix=prefix, recursive=True):
        name = obj.object_name
        if name.lower().endswith(".parquet"):
            objs.append(name)
            if max_files and len(objs) >= max_files:
                break
    return objs

def read_parquet_object(client: Minio, bucket: str, object_name: str) -> pd.DataFrame:
    resp = client.get_object(bucket, object_name)
    try:
        data = resp.read()
    finally:
        resp.close()
        resp.release_conn()

    return pd.read_parquet(io.BytesIO(data), engine="pyarrow")

def load_parquet_dataset_from_minio(bucket: str, prefix: str, max_files: int = 0) -> Tuple[pd.DataFrame, List[str]]:
    client = minio_client(CFG)
    object_names = list_parquet_objects(client, bucket=bucket, prefix=prefix, max_files=max_files)

    if not object_names:
        return pd.DataFrame(), []

    frames = []
    for name in object_names:
        df_part = read_parquet_object(client, bucket=bucket, object_name=name)
        frames.append(df_part)

    df_all = pd.concat(frames, ignore_index=True)
    return df_all, object_names


In [ ]:
df_raw, object_names = load_parquet_dataset_from_minio(BUCKET, PREFIX, MAX_FILES)

print(f"Loaded rows: {len(df_raw):,}")
print(f"Loaded columns: {df_raw.shape[1]:,}")
print(f"Parquet objects read: {len(object_names):,}")
df_raw.head()


## Data Structure and Schema
- Rows / columns
- Column list and dtypes
- Candidate key fields (expected: `exchange`, `ticker`, `trade_date`)
- Granularity check (one row per ticker-date)


In [ ]:
df_raw.dtypes

In [ ]:
df_raw.columns.tolist()

In [ ]:
# Candidate key uniqueness check
key_cols = [c for c in ["exchange", "ticker", "trade_date"] if c in df_raw.columns]
if len(key_cols) == 3:
    dup_keys = df_raw.duplicated(subset=key_cols).sum()
    print(f"Duplicate key rows (exchange,ticker,trade_date): {dup_keys:,}")
else:
    print(f"Key columns missing. Present key-like columns: {key_cols}")


## Data Quality Assessment
- Missingness by column
- Duplicates at full-row and key level
- Basic validity checks (non-negative volume; plausible ranges)
- Temporal coverage and gaps (per ticker)


In [ ]:
# Missingness
missing = df_raw.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(50)

In [ ]:
# Duplicate full rows
dup_full = df_raw.duplicated().sum()
dup_full

In [ ]:
# Basic validity: volume should be >= 0 if present
if "volume" in df_raw.columns:
    invalid_volume = (df_raw["volume"].dropna() < 0).sum()
    print(f"Invalid negative volume rows: {invalid_volume:,}")
else:
    print("No 'volume' column found.")

In [ ]:
# Temporal coverage summary
if "trade_date" in df_raw.columns:
    # Ensure parsable
    trade_date = pd.to_datetime(df_raw["trade_date"], errors="coerce")
    print("trade_date parse failures:", trade_date.isna().sum())
    if "ticker" in df_raw.columns:
        cov = (
            df_raw.assign(_trade_date=trade_date)
                 .groupby("ticker")["_trade_date"]
                 .agg(["min","max","nunique"])
                 .sort_values("nunique", ascending=False)
        )
        cov.head(20)
else:
    print("No 'trade_date' column found.")

## Univariate Analysis

### Numerical Features
- Distribution shape
- Central tendency and spread
- Potential outliers (identification only)

### Categorical Features
- Cardinality and dominant categories
- Rare categories

### Temporal Features
- Global coverage
- Ticker-level gaps (if applicable)


In [ ]:
# Identify feature types (heuristic)
num_cols = df_raw.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in df_raw.columns if c not in num_cols]

len(num_cols), len(cat_cols), num_cols[:20]

In [ ]:
# Numerical summary (robust view)
df_raw[num_cols].describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).T

In [ ]:
# Categorical cardinality (top 20)
card = pd.Series({c: df_raw[c].nunique(dropna=True) for c in cat_cols}).sort_values(ascending=False)
card.head(20)

In [ ]:
# Frequency check for key categorical columns (if present)
for c in ["exchange","ticker","vendor","currency"]:
    if c in df_raw.columns:
        display(df_raw[c].value_counts(dropna=False).head(20))

## Visual Inspection (Univariate)

Guidance:
- Use histograms for skewed measures (volume often heavy-tailed)
- Consider log-scale views for volume-like distributions


In [ ]:
# Histogram helper
def plot_hist(series: pd.Series, title: str, bins: int = 50):
    s = series.dropna()
    if s.empty:
        print(f"No data to plot for {title}")
        return
    plt.figure()
    plt.hist(s, bins=bins)
    plt.title(title)
    plt.xlabel(series.name)
    plt.ylabel("count")
    plt.show()

for c in [col for col in ["open","high","low","close","volume"] if col in df_raw.columns]:
    plot_hist(df_raw[c], f"Histogram: {c}")

## Bivariate Analysis

- Numerical vs numerical (correlation, scatter where appropriate)
- Categorical vs numerical (group summaries)
- Temporal behaviour (time series by ticker)


In [ ]:
# Correlation matrix for numeric fields (pairwise complete)
if len(num_cols) > 1:
    corr = df_raw[num_cols].corr(numeric_only=True)
    corr

In [ ]:
# Example: time series for a chosen ticker (override via env)
DEFAULT_TICKER = os.environ.get("ASX_TICKER", None)

ticker = DEFAULT_TICKER or (df_raw["ticker"].dropna().astype(str).iloc[0] if "ticker" in df_raw.columns and len(df_raw) else None)
ticker

In [ ]:
def plot_time_series_for_ticker(df: pd.DataFrame, ticker: str):
    if ticker is None:
        print("No ticker available to plot.")
        return
    if "ticker" not in df.columns or "trade_date" not in df.columns:
        print("Required columns missing: ticker/trade_date.")
        return

    d = df[df["ticker"].astype(str) == str(ticker)].copy()
    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce")
    d = d.dropna(subset=["trade_date"]).sort_values("trade_date")

    y_cols = [c for c in ["close","volume"] if c in d.columns]
    if not y_cols:
        print("No close/volume columns to plot.")
        return

    for y in y_cols:
        plt.figure()
        plt.plot(d["trade_date"], d[y])
        plt.title(f"{ticker} — {y} over time")
        plt.xlabel("trade_date")
        plt.ylabel(y)
        plt.show()

plot_time_series_for_ticker(df_raw, ticker)

In [ ]:
# Group summaries: typical price/volume by ticker (top N by row count)
if "ticker" in df_raw.columns:
    g = df_raw.groupby("ticker").size().sort_values(ascending=False).head(20)
    top_tickers = g.index.tolist()
    metrics = [c for c in ["close","volume"] if c in df_raw.columns]
    if metrics:
        summary = df_raw[df_raw["ticker"].isin(top_tickers)].groupby("ticker")[metrics].agg(["mean","median","std","min","max"])
        summary

## Multivariate Observations
- Interactions between price and volume behaviour
- Potential redundancy between OHLC fields
- Confounding effects (e.g., regime shifts over time)


In [ ]:
# Simple multivariate example: close vs volume scatter for selected ticker
if ticker and all(c in df_raw.columns for c in ["ticker","close","volume"]):
    d = df_raw[df_raw["ticker"].astype(str) == str(ticker)].dropna(subset=["close","volume"])
    if len(d):
        plt.figure()
        plt.scatter(d["close"], d["volume"])
        plt.title(f"{ticker} — volume vs close")
        plt.xlabel("close")
        plt.ylabel("volume")
        plt.show()


## Outlier Analysis (Identification Only)

At this stage:
- We *identify* potential extreme values (price spikes, volume spikes)
- We do **not** remove or cap values in the EDA notebook
- Treatment decisions are deferred to preprocessing, after assessing modelling objectives and leakage risks


In [ ]:
# Quick extreme value scan (top values)
for c in [col for col in ["close","volume"] if col in df_raw.columns]:
    print(f"Top 10 by {c}")
    display(df_raw[[col for col in ["trade_date","ticker",c] if col in df_raw.columns]].sort_values(c, ascending=False).head(10))


## Key Findings
- Summarise the most important structural and behavioural findings
- Note any data quality risks that need remediation


## Implications for Next Steps
- Identify required preprocessing steps (types, duplicates, missingness, outlier strategy)
- Identify candidate derived features (returns, volatility proxies, rolling windows)
- Decide whether sampling considerations apply (only once a target is defined)


## Notes and Open Questions
- Items requiring domain confirmation
- Any gaps in time coverage or unexpected vendor behaviour
- Any schema ambiguity that should be fixed upstream
